# RiskLens: Cost-Sensitive Credit Default & Fraud Detection Engine
### End-to-End Walkthrough: Expected Rupee Loss, PR-AUC, Threshold Optimization, SHAP, and PSI Drift

**Author:** Raunak Kumar (InfoSecIITR)  
**Domain:** FinTech Risk Underwriting & Fraud Intelligence (Tailored for American Express Risk Teams)

---

### Executive Summary & Commercial Motivation
Standard machine learning models optimize for **Accuracy** or Log-Loss, implicitly treating all errors equally. In institutional credit risk:
- **False Negative (Default/Fraud Approved):** Bank loses Exposure at Default (EAD) $\times$ Loss Given Default (LGD) $\approx$ **₹75,000+**.
- **False Positive (Good Applicant Declined):** Bank loses customer lifetime margin & acquisition cost $\approx$ **₹3,500**.

Because False Negatives are **~21x more expensive**, default 0.5 decision thresholds are catastrophic. RiskLens optimizes expected rupee loss, calibrates probabilities, explains decisions with SHAP, and continuously monitors population stability with PSI.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from risklens.data.generator import generate_train_oot_splits
from risklens.data.preprocessor import CreditRiskPreprocessor
from risklens.models.baseline import LogisticScorecardBaseline
from risklens.models.tree_models import LightGBMRiskClassifier
from risklens.models.calibrator import ProbabilityCalibrator
from risklens.cost.loss_matrix import RupeeLossCalculator
from risklens.cost.optimizer import ThresholdOptimizer
from risklens.cost.frontier import ApprovalFrontierEngine
from risklens.explainability.explainer import RiskLensExplainer
from risklens.monitoring.drift_detector import DriftDetector

print("RiskLens modules loaded successfully!")

## 1. High-Fidelity Data Generation
We generate 80,000 in-time records and 20,000 out-of-time (OOT) records with simulated macroeconomic stress.

In [ ]:
df_intime, df_oot = generate_train_oot_splits(n_train=40000, n_oot=10000, random_state=42)
print(f"In-Time cohort: {len(df_intime):,} records | Default rate: {df_intime['is_default'].mean():.2%}")
print(f"OOT cohort: {len(df_oot):,} records | Default rate: {df_oot['is_default'].mean():.2%}")
df_intime.head()

## 2. Preprocessing & Model Training
Fit robust scaler on numerical features and one-hot encoder on categorical attributes.

In [ ]:
prep = CreditRiskPreprocessor()
prep.fit(df_intime)
X = prep.transform(df_intime)
y = df_intime['is_default'].values
ead = df_intime['exposure_at_default'].values
lgd = df_intime['loss_given_default'].values

# Split 70% train, 15% calibration, 15% test
n_train = int(len(df_intime) * 0.70)
n_calib = int(len(df_intime) * 0.15)

X_train, y_train = X[:n_train], y[:n_train]
X_calib, y_calib = X[n_train:n_train+n_calib], y[n_train:n_train+n_calib]
X_test, y_test = X[n_train+n_calib:], y[n_train+n_calib:]
ead_test, lgd_test = ead[n_train+n_calib:], lgd[n_train+n_calib:]

# Train LightGBM
lgb = LightGBMRiskClassifier()
lgb.fit(X_train, y_train)

# Calibrate Probabilities
calib = ProbabilityCalibrator(method='isotonic')
calib.fit(lgb.model, X_calib, y_calib)
test_probs = calib.predict_proba(X_test)

print(f"Test cohort scored: {len(test_probs):,} applicants")

## 3. Cost-Sensitive Threshold Optimization
Sweep $\tau \in [0.005, 0.50]$ to find the cutoff minimizing expected rupee loss.

In [ ]:
optimizer = ThresholdOptimizer()
opt_res = optimizer.optimize_threshold(y_test, test_probs, ead=ead_test, lgd=lgd_test)

print(f"Optimal Threshold: τ* = {opt_res['optimal_threshold']:.3f}")
print(f"Loss at Naive 0.5: ₹{opt_res['naive_loss_rupees']:,.0f}")
print(f"Loss at Optimal Cutoff: ₹{opt_res['min_expected_loss_rupees']:,.0f}")
print(f"Rupee Savings: ₹{opt_res['total_rupee_savings']:,.0f} ({opt_res['pct_savings']:.1f}% reduction)")

## 4. The +4% Approval Expansion Frontier
Targeting 4% more approvals at the same expected-loss rate.

In [ ]:
frontier = ApprovalFrontierEngine()
base_scorecard = LogisticScorecardBaseline().fit(X_train, y_train)
base_probs = base_scorecard.predict_proba(X_test)

lift = frontier.compute_approval_expansion(
    y_true=y_test,
    baseline_prob=base_probs,
    risklens_prob=test_probs,
    ead=ead_test,
    lgd=lgd_test,
    baseline_approval_rate=0.78,
    target_lift_pct=4.0,
)

print("Baseline Approvals:", f"{lift['baseline']['approval_rate_pct']:.1f}%")
print("RiskLens Approvals:", f"{lift['risklens_expansion']['approval_rate_pct']:.1f}%")
print("Additional Good Accounts:", f"+{lift['lift_metrics']['additional_approved_accounts']:,}")
print("Incremental Revenue:", f"₹{lift['lift_metrics']['incremental_annual_revenue_rupees']:,.0f}")

## 5. SHAP Explainability & Adverse Action Notice
Local attribution breakdown for adverse decision compliance.

In [ ]:
explainer = RiskLensExplainer(lgb.model, prep.get_feature_names())
factors = explainer.explain_instance(X_test[0], top_k=5)
for f in factors:
    print(f"{f['feature']:<28} | Impact: {f['shap_value']:+.4f} | {f['direction']}")

## 6. Population Stability Index (PSI) Drift Monitoring
Calculate score PSI and feature PSI against out-of-time cohort.

In [ ]:
detector = DriftDetector()
oot_scores = calib.predict_proba(prep.transform(df_oot.iloc[:3000]))
audit = detector.run_drift_audit(
    baseline_df=df_intime.iloc[:5000],
    current_df=df_oot.iloc[:5000],
    baseline_scores=test_probs[:3000],
    current_scores=oot_scores,
)
print("Overall Audit Status:", audit['overall_status'])
print("Score PSI:", round(audit['score_psi'], 4), f"({audit['score_status']})")
audit['features_df'].head(8)